### Data Ingestion


In [4]:
from langchain_core.documents import Document


In [9]:
doc = Document(
    page_content = "This is main text context i am using to RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Muhammad waqas",
        "date_created":"2026-08-05"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Muhammad waqas', 'date_created': '2026-08-05'}, page_content='This is main text context i am using to RAG')

In [11]:
import os 

os.makedirs("../data/text_file", exist_ok = True)

In [13]:
sample_texts = {"../data/text_file/python_intro.txt":"""Python ek high-level, interpreted, general-purpose programming language hai. Iska syntax simple aur readable hota hai, isi wajah se beginners aur professional developers dono use karte hain.

Tum AI/ML aur RAG seekh rahe ho, isliye Python ki fundamentals strong karna important hai.
1. Python kya hai?
Python se hum different types ke software aur applications bana sakte hain:
- AI & Machine Learning: Scikit-learn, PyTorch, TensorFlow
- Data Science: Pandas, NumPy, Matplotlib
- Backend Development: FastAPI, Django, Flask
- Generative AI: LangChain, LlamaIndex
- Automation: Files, Excel, web automation""",

    "../data/text_file/machine_learning_intro":""""Machine Learning (ML) – Introduction
Machine Learning, Artificial Intelligence (AI) ki ek important branch hai. Ismein hum computer ko data se patterns seekhna sikhate hain, taake woh naye data par predictions ya decisions kar sake.

Simple words mein:

Traditional Programming: Hum computer ko rules dete hain aur woh un rules ke mutabiq output deta hai.

Machine Learning: Hum computer ko data aur correct examples dete hain, aur algorithm unmein patterns learn karta hai.
1. AI vs ML vs DL
In teen terms ko samajhna bohat important hai.
Artificial Intelligence (AI)

Machines performing intelligent tasks

Machine Learning (ML)

Learning patterns from data

Deep Learning (DL)

Learning using neural networks


Deep Learning, Machine Learning ka subset hai aur Machine Learning, AI ka subset hai."""


}

for filepath,content in sample_texts.items():
    with open(filepath, 'w',encoding='utf-8')as f:
        f.write(content)


print('Sapmle texts file created!')

Sapmle texts file created!


In [7]:
### Text Loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader('../data/text_file/machine_learning_intro', encoding = "utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_file/machine_learning_intro'}, page_content='"Machine Learning (ML) – Introduction\nMachine Learning, Artificial Intelligence (AI) ki ek important branch hai. Ismein hum computer ko data se patterns seekhna sikhate hain, taake woh naye data par predictions ya decisions kar sake.\n\nSimple words mein:\n\nTraditional Programming: Hum computer ko rules dete hain aur woh un rules ke mutabiq output deta hai.\n\nMachine Learning: Hum computer ko data aur correct examples dete hain, aur algorithm unmein patterns learn karta hai.\n1. AI vs ML vs DL\nIn teen terms ko samajhna bohat important hai.\nArtificial Intelligence (AI)\n\nMachines performing intelligent tasks\n\nMachine Learning (ML)\n\nLearning patterns from data\n\nDeep Learning (DL)\n\nLearning using neural networks\n\n\nDeep Learning, Machine Learning ka subset hai aur Machine Learning, AI ka subset hai.')]


In [13]:
### Directory loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_file",
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=True
)


documents = loader.load()
print('Total document:',len(documents))

Total document: 1


In [16]:
### Directory loader pdf
from langchain_community.document_loaders import PyPDFLoader , PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="*.pdf", 
    loader_cls=PyMuPDFLoader,
    show_progress=True
)


pdf_document = dir_loader.load()
pdf_document

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:03<00:00,  1.02s/it]


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'file_path': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='RESEARCH REPORT\nArtificial Intelligence:\nFoundations, Technologies,\nand Society\nA research report on AI history, major fields, generative models, applications,\nand governance\nPrepared by\n[Your Name]\nResearch date\n17 November 2026\nDocument type\nEducational research report\nAbstract\nThis report surveys artificial intelligence (AI) as a broad field concerned with computational\nsystems that perform tasks such as perception, reasoning, language processing, planning, and\ndecision support. It traces key historical milestones, explains major technical branches, introduces\nneural n

In [17]:
print("Total PDF pages:", len(pdf_document))

print("\nFirst page text:")
print(pdf_document[0].page_content[:500])

print("\nMetadata:")
print(pdf_document[0].metadata)

Total PDF pages: 21

First page text:
RESEARCH REPORT
Artificial Intelligence:
Foundations, Technologies,
and Society
A research report on AI history, major fields, generative models, applications,
and governance
Prepared by
[Your Name]
Research date
17 November 2026
Document type
Educational research report
Abstract
This report surveys artificial intelligence (AI) as a broad field concerned with computational
systems that perform tasks such as perception, reasoning, language processing, planning, and
decision support. It traces key

Metadata:
{'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'file_path': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}


### Embeding And VectorstoreDB

In [28]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Tuple,Dict,Any 
from sentence_transformers import SentenceTransformer
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
class EmbeddingManager:
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model() 

    def _load_model(self):
        try:
            print(f"Load Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") 
        except Exception as e:
            print(f"Error loaded model: {self.model_name}:{e}")
            raise 

    def _generate_embaddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Modelnot loaded") 

        print(f"Generation Embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embedding with shape: {embeddings.shape}") 
        return embeddings         
    


embedding_manager = EmbeddingManager()
embedding_manager

Load Embedding model: all-MiniLM-L6-v2


c:\Users\User\Desktop\Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\User\Desktop\Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning

Model Loaded Successfully. Embedding dimension: 384


C:\Users\User\AppData\Local\Temp\ipykernel_12760\3277505196.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store


In [ ]:
class VectorStore:
    def __init__(self,collection_name: str = "pdf_documents", persist_directory: str = "../data/Vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)


            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF Document Embedding for RAG"}
            )
            print(f"Vector store initialize. collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error Initializing Vector Store: {e}")
            raise         



    def add_documents(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        #Prepare the data of chromadb
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embeddings_vector) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepear metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #documents content
            documents_text.append(doc.page_content)

            #Embeddings
            embeddings_list.append(embeddings_vector.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas = metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store") 
            print(f"Total  documents in collection: {self.collection.count()}") 

        except Exception as e:
            print(f"Error adding document  to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore               

        

Vector store initialize. collection: pdf_documents
Existing documents in collection: 0


In [21]:
texts = [doc.page_content for doc in pdf_document]

embeddings = embedding_manager._generate_embaddings(texts)

Generation Embedding for 21 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.86s/it]

Generated embedding with shape: (21, 384)


In [22]:
print(embeddings.shape)

(21, 384)


In [23]:
vectorstore.add_documents(pdf_document, embeddings)

Adding 21 documents to vector store...
Successfully added 21 documents to vector store
Total  documents in collection: 21


In [25]:
print("Total Chunks:", vectorstore.collection.count())

Total Chunks: 21


In [27]:
all_chunks = vectorstore.collection.get(
    include=["documents", "metadatas"]
)

for i, chunk in enumerate(all_chunks["documents"]):

    print(f"\n{'='*70}")
    print(f"CHUNK {i+1}")
    print(f"{'='*70}")

    print(chunk)


CHUNK 1
RESEARCH REPORT
Artificial Intelligence:
Foundations, Technologies,
and Society
A research report on AI history, major fields, generative models, applications,
and governance
Prepared by
[Your Name]
Research date
17 November 2026
Document type
Educational research report
Abstract
This report surveys artificial intelligence (AI) as a broad field concerned with computational
systems that perform tasks such as perception, reasoning, language processing, planning, and
decision support. It traces key historical milestones, explains major technical branches, introduces
neural networks and transformer models, and discusses applications, system development, risks,
ethics, and governance. It is intended as an accessible academic introduction rather than a claim
that current systems possess human consciousness or general intelligence.
Keywords: Artificial Intelligence, AI, machine learning, deep learning, neural networks, transformers, NLP, computer
vision, generative AI, AI governance


### Spliting into Chunks


In [29]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [30]:
chunks = text_splitter.split_documents(pdf_document)

In [31]:
for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i+1} ---")
    print(chunk.page_content)


--- CHUNK 1 ---
RESEARCH REPORT
Artificial Intelligence:
Foundations, Technologies,
and Society
A research report on AI history, major fields, generative models, applications,
and governance
Prepared by
[Your Name]
Research date
17 November 2026
Document type
Educational research report
Abstract
This report surveys artificial intelligence (AI) as a broad field concerned with computational
systems that perform tasks such as perception, reasoning, language processing, planning, and

--- CHUNK 2 ---
decision support. It traces key historical milestones, explains major technical branches, introduces
neural networks and transformer models, and discusses applications, system development, risks,
ethics, and governance. It is intended as an accessible academic introduction rather than a claim
that current systems possess human consciousness or general intelligence.
Keywords: Artificial Intelligence, AI, machine learning, deep learning, neural networks, transformers, NLP, computer

--- CHUNK 3

In [32]:
texts = [chunk.page_content for chunk in chunks]

chunk_embeddings = embedding_manager._generate_embaddings(texts)

Generation Embedding for 100 texts...


Batches: 100%|██████████| 4/4 [00:03<00:00,  1.04it/s]

Generated embedding with shape: (100, 384)


In [33]:
chunk_vectorstore = VectorStore(
    collection_name="pdf_chunks"
)

Vector store initialize. collection: pdf_chunks
Existing documents in collection: 0


In [34]:
chunk_vectorstore.add_documents(
    chunks,
    chunk_embeddings
)

Adding 100 documents to vector store...
Successfully added 100 documents to vector store
Total  documents in collection: 100


In [35]:
print(chunk_vectorstore.collection.count())

100


In [37]:
chunks

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'file_path': '..\\data\\pdf\\Artificial_Intelligence_Research_Report.pdf', 'total_pages': 7, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='RESEARCH REPORT\nArtificial Intelligence:\nFoundations, Technologies,\nand Society\nA research report on AI history, major fields, generative models, applications,\nand governance\nPrepared by\n[Your Name]\nResearch date\n17 November 2026\nDocument type\nEducational research report\nAbstract\nThis report surveys artificial intelligence (AI) as a broad field concerned with computational\nsystems that perform tasks such as perception, reasoning, language processing, planning, and'),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Artificial_

In [46]:
print(chunk_vectorstore.collection.name)
print(chunk_vectorstore.collection.count())

pdf_chunks
100


In [47]:
print(vectorstore.collection.name)
print(vectorstore.collection.count())

pdf_documents
21


In [48]:
all_chunks = chunk_vectorstore.collection.get(
    include=["documents", "metadatas"]
)

print("Total:", len(all_chunks["ids"]))

print(all_chunks["ids"][:5])
print(all_chunks["ids"][100:105])

Total: 100
['doce2b16bf0_0', 'doc4d9c5477_1', 'doc5d90bf01_2', 'docac29ffb1_3', 'docad2460ce_4']
[]


In [42]:
chunk_vectorstore.client.delete_collection("pdf_chunks")

In [43]:
chunk_vectorstore.collection = (
    chunk_vectorstore.client.create_collection(
        name="pdf_chunks",
        metadata={"description": "PDF Document Embedding for RAG"}
    )
)

In [44]:
chunk_vectorstore.add_documents(
    chunks,
    chunk_embeddings
)

Adding 100 documents to vector store...
Successfully added 100 documents to vector store
Total  documents in collection: 100


In [45]:
print(chunk_vectorstore.collection.count())

100


### Retriever Pipeline From VectorStore

In [53]:
class RAGRetriever:
    def __init__(self,vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0)-> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score_threshold: {score_threshold}")    

        query_embedding = self.embedding_manager._generate_embaddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })  
            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs
                    

        except Exception as e :
            print(f"Error during retrieval: {e}")
            return []    

rag_retriever = RAGRetriever(chunk_vectorstore, embedding_manager)
rag_retriever        

In [54]:
results = rag_retriever.retrieve(
    query="What is Artificial Intelligence?",
    top_k=5
)

Retrieving documents for query: 'What is Artificial Intelligence?'
Top K: 5, score_threshold: 0.0
Generation Embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.58it/s]

Generated embedding with shape: (1, 384)
Retrieved 5 documents (after filtering)


In [55]:
for doc in results:
    print("\n", "="*60)
    print("Rank:", doc["rank"])
    print("Distance:", doc["distance"])
    print("Content:", doc["content"])


Rank: 1
Distance: 0.3454810082912445
Content: Artificial Intelligence Research Report
Page 3
Introduction and definition
Artificial intelligence is a broad research and engineering field focused on creating computational
systems that perform tasks commonly associated with intelligent behavior. Examples include
recognizing patterns, understanding or generating language, planning actions, solving problems,
and supporting decisions.
AI is not one algorithm or product. It includes symbolic methods, statistical learning, optimization,

Rank: 2
Distance: 0.5902819633483887
Content: evaluation.
AI, machine learning, and deep learning
Artificial intelligence (AI) is the broad field concerned with building systems that perform tasks
associated with intelligent behavior. Machine learning is one approach within AI: instead of
specifying every rule by hand, developers train models from examples or feedback. Deep learning
is a family of machine-learning methods based on neural networks with multip